# Delhi smoke: do Punjab crop fires explain Delhi's winter air?

**The question:** every October and November, farmers in Punjab and Haryana burn rice stubble. Delhi's air collapses. But does *fire count alone* explain how bad a day gets?

**The hypothesis:** no. Smoke only reaches Delhi when the wind blows from the north-west. The same number of fires can produce a terrible day or a tolerable one depending on wind direction.

Run the cells in order. Each stage produces a chart worth looking at, so you can stop at any point and still have something.

| Stage | What you get |
|---|---|
| 1 | Five burning seasons of satellite fire detections |
| 2 | Fires plotted against Delhi PM2.5 |
| 3 | The wind test: does transport beat raw fire count? |

In [ ]:
import glob, io, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

pd.set_option('display.width', 120)
plt.rcParams['figure.figsize'] = (13, 4.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('ready')

## Stage 1 — the fires

Run the next cell and upload **all** your FIRMS CSVs at once (`fire_archive_SV-C2_*.csv`) plus **`city_day.csv`**. You can select multiple files in the dialog.

If you are not on Colab, skip this cell and put the files in the same folder as the notebook instead.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    print('\nuploaded:', list(uploaded))
except ImportError:
    print('Not on Colab - reading files from the current folder instead.')

In [ ]:
# --- geography -----------------------------------------------------------
# Punjab/Haryana bounding box, and Delhi.
WEST, SOUTH, EAST, NORTH = 73.5, 27.5, 77.8, 32.6
DELHI_LAT, DELHI_LON = 28.6139, 77.2090
SOURCE_LAT, SOURCE_LON = 30.4, 75.6   # centre of the burning belt


def load_fires(pattern='*SV-C2*.csv'):
    """Read every FIRMS export in the folder and stack them."""
    paths = sorted(glob.glob(pattern)) or sorted(glob.glob('fire_*.csv'))
    if not paths:
        raise FileNotFoundError(
            'No FIRMS CSVs found. Expected files named like '
            'fire_archive_SV-C2_123456.csv in this folder.'
        )
    frames = [pd.read_csv(p) for p in paths]
    df = pd.concat(frames, ignore_index=True)
    print(f'read {len(paths)} file(s), {len(df):,} raw detections')
    return df


def clean_fires(df):
    """Parse dates, clip to the box, drop low-confidence detections."""
    out = df.copy()
    out.columns = [c.strip().lower() for c in out.columns]
    date_col = 'acq_date' if 'acq_date' in out.columns else 'date'
    out['date'] = pd.to_datetime(out[date_col], errors='coerce').dt.normalize()
    out = out.dropna(subset=['date'])

    out = out[out['latitude'].between(SOUTH, NORTH) &
              out['longitude'].between(WEST, EAST)]

    # VIIRS confidence is l/n/h (or low/nominal/high); MODIS is 0-100.
    # Decide on content, not dtype: pandas infers text columns differently
    # across versions, and a dtype check silently drops every VIIRS row.
    if 'confidence' in out.columns:
        numeric = pd.to_numeric(out['confidence'], errors='coerce')
        if numeric.notna().mean() > 0.5:
            keep = numeric >= 30
        else:
            first = out['confidence'].astype(str).str.strip().str.lower().str[0]
            keep = first.isin({'n', 'h'})
        out = out[keep.fillna(False).astype(bool)].copy()

    if 'frp' in out.columns:
        out['frp'] = pd.to_numeric(out['frp'], errors='coerce')

    print(f'{len(out):,} detections kept after cleaning')
    return out.reset_index(drop=True)


def daily_fires(df):
    """One row per day: how many fires, and how intense."""
    g = df.groupby('date')
    out = pd.DataFrame({'fire_count': g.size()})
    if 'frp' in df.columns:
        out['frp_sum'] = g['frp'].sum()
    return out.reset_index().sort_values('date')


fires_raw = load_fires()
fires = clean_fires(fires_raw)
fires_daily = daily_fires(fires)
fires_daily.head()

In [ ]:
# Fires per day, one panel per burning season.
fires_daily['year'] = fires_daily['date'].dt.year
years = sorted(fires_daily['year'].unique())

fig, axes = plt.subplots(len(years), 1, figsize=(13, 2.1 * len(years)), sharex=False)
axes = np.atleast_1d(axes)
for ax, yr in zip(axes, years):
    sub = fires_daily[fires_daily['year'] == yr]
    ax.fill_between(sub['date'], sub['fire_count'], alpha=0.75, color='#c1440e')
    ax.set_ylabel(str(yr))
    ax.margins(x=0)
axes[0].set_title('Fire detections per day, Punjab + Haryana (VIIRS S-NPP)')
plt.tight_layout()
plt.show()

print(fires_daily.groupby('year')['fire_count'].agg(['sum', 'max']).rename(
    columns={'sum': 'total_detections', 'max': 'worst_single_day'}))

## Stage 2 — Delhi's air

Now bring in PM2.5 and see how closely it tracks the fires.

In [ ]:
aq = pd.read_csv('city_day.csv')
aq.columns = [c.strip() for c in aq.columns]
aq['Date'] = pd.to_datetime(aq['Date'], errors='coerce')

delhi = (aq[aq['City'].str.strip().str.lower() == 'delhi']
         .rename(columns={'Date': 'date', 'PM2.5': 'pm25', 'AQI': 'aqi'})
         [['date', 'pm25', 'aqi']]
         .dropna(subset=['date'])
         .sort_values('date'))

print(f"Delhi rows: {len(delhi):,}")
print(f"date range: {delhi['date'].min().date()} to {delhi['date'].max().date()}")
print(f"missing PM2.5: {delhi['pm25'].isna().sum()} days")
delhi.head()

In [ ]:
# Join. A day inside the study window with no detections is a real zero -
# the satellite passed and saw nothing - so fill with 0, not NaN.
start, end = fires_daily['date'].min(), fires_daily['date'].max()

df = (pd.DataFrame({'date': pd.date_range(start, end, freq='D')})
      .merge(fires_daily.drop(columns='year'), on='date', how='left')
      .merge(delhi, on='date', how='left'))
df[['fire_count', 'frp_sum']] = df[['fire_count', 'frp_sum']].fillna(0)

# Keep only the burning season, and only days where we actually know the air.
df = df[df['date'].dt.month.isin([9, 10, 11, 12])]
df = df.dropna(subset=['pm25']).reset_index(drop=True)
print(f'{len(df):,} usable days across {df["date"].dt.year.nunique()} seasons')
df.head()

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 4.5))
ax1.fill_between(df['date'], df['fire_count'], alpha=0.55, color='#c1440e', label='fire detections')
ax1.set_ylabel('fire detections / day', color='#c1440e')
ax2 = ax1.twinx()
ax2.plot(df['date'], df['pm25'], lw=1.1, color='#1f3b57', label='Delhi PM2.5')
ax2.set_ylabel('Delhi PM2.5 (ug/m3)', color='#1f3b57')
ax2.grid(False)
ax1.set_title('Punjab/Haryana fires vs Delhi PM2.5, burning seasons only')
plt.tight_layout(); plt.show()

print('correlation, fire_count vs PM2.5:', round(df['fire_count'].corr(df['pm25']), 3))

## Stage 3 — the wind test

This is the part that makes the project yours.

Punjab and Haryana sit **north-west** of Delhi. Smoke only arrives when the wind blows *from* that direction. So fire count on its own should be a weaker predictor than fire count **conditioned on wind**.

Wind comes from Open-Meteo's reanalysis archive — free, no API key.

In [ ]:
def bearing(lat1, lon1, lat2, lon2):
    """Initial great-circle bearing from point 1 to point 2, degrees [0,360)."""
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dl = np.radians(lon2 - lon1)
    y = np.sin(dl) * np.cos(p2)
    x = np.cos(p1) * np.sin(p2) - np.sin(p1) * np.cos(p2) * np.cos(dl)
    return (np.degrees(np.arctan2(y, x)) + 360) % 360


SOURCE_BEARING = bearing(DELHI_LAT, DELHI_LON, SOURCE_LAT, SOURCE_LON)
print(f'Fire belt lies {SOURCE_BEARING:.1f} deg from Delhi (north-west, as expected)')


def circular_mean(deg, weights=None):
    """Wind direction is circular: the mean of 350 and 10 is 0, not 180.
    Averaging as unit vectors is the only correct way."""
    deg = np.asarray(deg, float)
    m = ~np.isnan(deg)
    if not m.any():
        return np.nan
    deg = deg[m]
    w = np.ones_like(deg) if weights is None else np.asarray(weights, float)[m]
    r = np.radians(deg)
    return (np.degrees(np.arctan2((w * np.sin(r)).sum(), (w * np.cos(r)).sum())) + 360) % 360


print('sanity check - mean of 350 and 10 degrees:')
print(f'  circular (correct): {circular_mean([350, 10]):.1f}')
print(f'  arithmetic (wrong): {np.mean([350, 10]):.1f}')

In [ ]:
def fetch_weather(start, end):
    r = requests.get(
        'https://archive-api.open-meteo.com/v1/archive',
        params={'latitude': DELHI_LAT, 'longitude': DELHI_LON,
                'start_date': str(start.date()), 'end_date': str(end.date()),
                'hourly': 'wind_speed_10m,wind_direction_10m,temperature_2m,precipitation',
                'timezone': 'Asia/Kolkata'}, timeout=120)
    r.raise_for_status()
    h = pd.DataFrame(r.json()['hourly'])
    h['time'] = pd.to_datetime(h['time'])
    h['date'] = h['time'].dt.normalize()

    rows = []
    for day, g in h.groupby('date'):
        rows.append({
            'date': day,
            'wind_speed': g['wind_speed_10m'].mean(),
            'wind_dir': circular_mean(g['wind_direction_10m'], g['wind_speed_10m']),
            'temp': g['temperature_2m'].mean(),
            'rain': g['precipitation'].sum(),
        })
    return pd.DataFrame(rows)


wx = fetch_weather(df['date'].min(), df['date'].max())
print(f'{len(wx):,} days of weather')

df = df.merge(wx, on='date', how='left')

# How aligned is the wind with the fire belt? 1 = straight from the fires.
delta = np.abs(df['wind_dir'] - SOURCE_BEARING) % 360
delta = np.where(delta > 180, 360 - delta, delta)
df['alignment'] = np.clip(np.cos(np.radians(delta)), 0, 1)

# The headline feature: fires only matter when the wind points at us.
df['transport_index'] = df['fire_count'] * df['alignment']
df['transport_lag1'] = df['transport_index'].shift(1)
df['stagnation'] = 1 / (1 + df['wind_speed'])
df[['date', 'fire_count', 'wind_dir', 'alignment', 'transport_index', 'pm25']].head(10)

In [ ]:
# THE TEST: does conditioning on wind beat raw fire count?
for name in ['fire_count', 'frp_sum', 'transport_index', 'transport_lag1']:
    print(f'{name:>18} vs PM2.5:  r = {df[name].corr(df["pm25"]):.3f}')

# Same fires, different wind: split high-fire days by wind alignment.
busy = df[df['fire_count'] > df['fire_count'].quantile(0.75)]
aligned = busy[busy['alignment'] > 0.5]['pm25']
cross = busy[busy['alignment'] <= 0.5]['pm25']

print(f'\nHigh-fire days only ({len(busy)} days):')
print(f'  wind from the fires   : median PM2.5 {aligned.median():.0f}  (n={len(aligned)})')
print(f'  wind across or away   : median PM2.5 {cross.median():.0f}  (n={len(cross)})')

from scipy import stats
if len(aligned) > 5 and len(cross) > 5:
    u, p = stats.mannwhitneyu(aligned.dropna(), cross.dropna(), alternative='greater')
    print(f'  Mann-Whitney U, one-sided p = {p:.2e}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(df['fire_count'], df['pm25'], c=df['alignment'],
                cmap='RdYlBu_r', s=18, alpha=0.85)
ax.set_xlabel('fire detections that day')
ax.set_ylabel('Delhi PM2.5')
ax.set_title('Same fires, different wind')
plt.colorbar(sc, label='wind alignment (1 = blowing from the fires)')
plt.tight_layout(); plt.show()

## What to look for

If the hypothesis holds, `transport_index` correlates more strongly with PM2.5 than `fire_count` does, and on the busiest fire days the aligned-wind group has visibly worse air than the cross-wind group.

**If it doesn't hold, that is still a result** — and a more interesting one to write up honestly than a fudged positive. Report what you find, not what you hoped for. Possible reasons worth investigating: Delhi's winter pollution is dominated by local sources (traffic, industry, heating) and boundary-layer collapse rather than transport; or daily averaging is too coarse and the signal lives in hourly data.